<a href="https://colab.research.google.com/github/zohare11/butterflai2/blob/main/weeks/week_01/00_setup_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦋 ButterflAI 2.0 — Setup check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SwRI-IDEA-Lab/butterflai2/blob/main/weeks/week_01/00_setup_check.ipynb)

Run this notebook once before the first class to confirm your environment works.

1. Run the **setup cell**. On Colab it downloads the course repository and installs the packages we use (about a minute).
2. Run the **check cell**. If everything is in place, you'll get a welcome message.

If something fails, copy the full error message and bring it to class or post it to the course channel.

In [1]:
# ── ButterflAI 2.0 standard setup — run this first ─────────────────────────
import os, sys, random, subprocess
from pathlib import Path

REPO_URL = "https://github.com/SwRI-IDEA-Lab/butterflai2.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = Path("/content/butterflai2")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO / "requirements.txt")], check=True)
else:
    # Local: walk up from the notebook's folder to the repository root.
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "legacy" / "butterflAI_model.py").exists()), None)
    if REPO is None:
        raise RuntimeError("Could not find the butterflai2 repository root. "
                           "Open this notebook from inside your clone.")

DATA = REPO / "data"
DERIVED = DATA / "derived"
DERIVED.mkdir(exist_ok=True)
LEGACY = REPO / "legacy"
sys.path.insert(0, str(LEGACY))   # enables: from butterflAI_model import ButterflAIModel

SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available()
                          else "mps" if torch.backends.mps.is_available() else "cpu")
except ImportError:
    DEVICE = "cpu"

print(f"🦋 ButterflAI 2.0 | {'Colab' if IN_COLAB else 'local'} | repo: {REPO} | device: {DEVICE}")


🦋 ButterflAI 2.0 | Colab | repo: /content/butterflai2 | device: cpu


In [2]:
# ── Check that everything we need is in place ──────────────────────────────
import importlib

problems = []

# 1. Packages
for pkg in ["numpy", "pandas", "scipy", "matplotlib", "torch", "lightning", "wandb"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"  ✓ {pkg:<11} {getattr(mod, '__version__', '')}")
    except ImportError:
        print(f"  ✗ {pkg:<11} MISSING")
        problems.append(f"package '{pkg}' is not installed")

# 2. Course data
catalog = DATA / "composite_sunspot_groups_daily_measurements_10_23.csv"
if catalog.exists():
    print(f"  ✓ data        {catalog.name} ({catalog.stat().st_size / 1e6:.1f} MB)")
else:
    print("  ✗ data        sunspot catalog not found")
    problems.append(f"missing data file: {catalog}")

# 3. The ButterflAI 1.0 model we will use in Week 1
try:
    from butterflAI_model import ButterflAIModel
    ButterflAIModel()
    print("  ✓ legacy      ButterflAI 1.0 model loads")
except Exception as e:
    print(f"  ✗ legacy      {e}")
    problems.append("the ButterflAI 1.0 model did not load")

# 4. Hardware (a GPU is not needed until Week 5)
print(f"  · device      {DEVICE}")

print()
if problems:
    print("Almost there — fix these before class:")
    for p in problems:
        print("   •", p)
else:
    print("🦋  Welcome to ButterflAI 2.0!  Your setup is ready.")
    print("    See you in class — we'll start by counting sunspots.")


  ✓ numpy       2.1.3
  ✓ pandas      2.2.3
  ✓ scipy       1.16.3
  ✓ matplotlib  3.10.0
  ✓ torch       2.11.0+cpu
  ✓ lightning   2.6.6
  ✓ wandb       0.28.1
  ✓ data        composite_sunspot_groups_daily_measurements_10_23.csv (19.8 MB)
  ✓ legacy      ButterflAI 1.0 model loads
  · device      cpu

🦋  Welcome to ButterflAI 2.0!  Your setup is ready.
    See you in class — we'll start by counting sunspots.
